In [0]:
df = spark.read.table(
    "finops_catalog_2.finops.gold_anomaly_features"
)

In [0]:
feature_df = df.select(
    "date","anomaly_score","business_unit", "budget_amount","department",
    "service","net_cost", "usage_quantity",
    "budget_utilization_pct","cost_variance_7d_pct","cost_variance_30d_pct"
)

display(feature_df.limit(10))


### Cost-to-Budget Ratio

In [0]:

from pyspark.sql.functions import *

feature_df = feature_df.withColumn(
    "cost_budget_ratio",
    round(col("net_cost") / col("budget_amount"),4)
)

feature_df = feature_df.withColumn(
    "cost_budget_pct",
    round(col("cost_budget_ratio") * 100, 2)
)

feature_df.select("cost_budget_ratio","cost_budget_pct").show()

### Cost Per Usage Unit

In [0]:
feature_df = feature_df.withColumn(
    "cost_per_unit",
    round(
        col("net_cost") / col("usage_quantity"),
        4
    )
)

### Savings Percentage

In [0]:
full_df = spark.read.table(
    "silver_cloud_budget"
)

feature_df = feature_df.join(
    full_df.select(
        "date",
        "account_id",
        "reserved_savings",
        "savings_plan_savings",
        "spot_savings",
        "net_cost"
    ),
    on=["date", "net_cost"],
    how="left"
)

### Total Savings Feature

In [0]:
feature_df = feature_df.withColumn(
    "total_savings",
    col("reserved_savings")
    + col("savings_plan_savings")
    + col("spot_savings")
)

### Day Features

In [0]:
feature_df = (
    feature_df
    .withColumn(
        "day_of_week",
        dayofweek("date")
    )
    .withColumn(
        "week_of_year",
        weekofyear("date")
    )
    .withColumn(
        "month",
        month("date")
    )
)

### High Budget Utilization Flag

In [0]:
%sql
USE CATALOG finops_catalog_2;
USE SCHEMA finops;

In [0]:
feature_df = feature_df.withColumn(
    "high_budget_flag",
    when(
        col("budget_utilization_pct") > 0.0200,
        1
    ).otherwise(0)
)

### High Variance Flag

In [0]:
display(feature_df[feature_df["cost_variance_7d_pct"]>0.3490].count())

In [0]:
feature_df = feature_df.withColumn(
    "high_variance_flag",
    when(
        abs(col("cost_variance_7d_pct")) > 0.3490,
        1
    ).otherwise(0)
)

In [0]:
%sql
SELECT current_catalog();

In [0]:
spark.sql("USE CATALOG finops_catalog_2")
spark.sql("USE SCHEMA finops")

In [0]:
pdf = feature_df.toPandas()

### Final Feature Set

In [0]:
ml_features = feature_df.select(
    "date",
    "net_cost",
    "usage_quantity",
    "anomaly_score",
    "budget_amount",
    "budget_utilization_pct",
    "cost_variance_7d_pct",
    "cost_variance_30d_pct",
    "cost_budget_pct",
    "cost_per_unit",
    "total_savings",
    "high_budget_flag",
    "high_variance_flag"
)

### Save Engineered Features

In [0]:
ml_features.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("finops_catalog_2.finops.ml_features")



In [0]:
df = spark.read.table(
    "finops_catalog_2.finops.ml_features"
)

pdf = df.toPandas()

pdf.to_csv(
    "/Workspace/Users/omkarmhetre800@gmail.com/Cloud-Cost-Anomaly-Detection/assets/ml_features.csv",
    index=False
)